# Can an LLM recommend independently from a user's ratings?

This is a **direct-generation** comparison. GPT-6 Luna receives each user's full training
rating history and generates ten movie recommendations using its learned movie knowledge.
Unlike [notebook 11](11_llm_reranking_experiment.ipynb), it receives **no candidate list**, catalog,
model scores, plot descriptions, cast metadata, IMDb averages, dataset name, user IDs, or movie IDs.
It uses no browsing tools and is not fine-tuned on the project data.

Use the same 100 seeded validation users from notebook 11. Their full training histories are
supplied chronologically as `[movie title including year, star rating]` rows, without truncation.
The models are compared against **all held-out validation likes**, retaining the existing positive
threshold, Recall@10 and binary NDCG@10 definitions. Test feedback is never read.

## What the comparison measures

The LLM's pretraining knowledge is encoded in its parameters; it does not retrieve or inspect its
original training corpus. It can infer preferences from rated movies and recommend independently.
Its output is evaluated against the same recorded future likes as the dedicated models.

EASE and the IMDb hybrid full-catalog baselines are reused from notebook 11, with their recorded
settings and the same users. This experiment changes both **generation mode** and **history length**
relative to notebook 11, so differences between the two LLM results cannot isolate either factor.

## Fixed grounding and scoring rules

Ask for ten titles and original release years. Only **after generation**, match them to the complete
MovieLens metadata catalog using normalized punctuation/accents, MovieLens aliases, and exact years.
The matcher never consults validation labels, uses no fuzzy title matching, and never guesses among
ambiguous matches. Then apply the same training-observed candidate universe and seen exclusions
used by the dedicated models.

Unmatched, ambiguous, duplicate, already-rated, and outside-training-catalog suggestions consume
their original rank slots and contribute no hits. Do not compact the remaining list or ask for
replacement suggestions. Invalid/incomplete responses yield zero hits and users are never dropped.
Audit every emitted title, match status, original rank, and validation hit.

**An unmatched or out-of-catalog suggestion is not necessarily a bad recommendation or a hallucination.**
MovieLens cannot establish relevance for movies it does not contain or that the user did not rate.
These metrics measure recovery of recorded held-out likes; they do not fully evaluate open-world
movie recommendation quality. Current-model knowledge and repeated validation comparisons also
limit historical and generalization claims.

## Budget and access

Read the API key from the local ignored `.env` or environment. `RUN_LIVE = False` below makes no
API requests and shows saved responses. Live requests share notebook 11's **original persistent $1
reservation ledger and lock**, preserving all previous reservations. There are no automatic retries.
The complete pending batch must fit the remaining reservation allowance before any new call.
Never delete the ledger to bypass the ceiling. Reservations are conservative client-side estimates
at the recorded rates, not a provider billing limit. No OpenAI SDK installation is required.

Prerequisite: run notebook 11's preparation to freeze users and full-catalog baseline results.
Live completion of notebook 11 is not required to prepare this experiment.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "scripts/llm_direct_recommendations.py").is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from scripts.llm_direct_recommendations import run_direct

RUN_LIVE = False  # Explicitly enable only to make uncached direct-generation API requests.
summary, report = run_direct(project_root, live=RUN_LIVE)
output_dir = project_root / "data/processed/per_user_chronological_60_20_20_v1/notebook12_llm_direct"
display(summary.round(6))

Prepared 100 full-history users; new reservations up to $0.1593; already reserved $0.8287.
                   model  users  mean_recall_at_10  mean_ndcg_at_10
       EASE full catalog    100           0.143138         0.149123
IMDb hybrid full catalog    100           0.157483         0.150806
 LLM direct full history    100           0.077828         0.093032
{
  "complete": true,
  "planned_users": 100,
  "attempted_users": 100,
  "valid_format_users": 100,
  "slot_status_counts": {
    "eligible": 910,
    "unmatched_title_year": 40,
    "already_rated": 34,
    "outside_training_catalog": 15,
    "duplicate": 1
  },
  "estimated_token_cost_usd": 0.0212481,
  "cumulative_reserved_usd_both_experiments": 0.82873075,
  "plan_sha256": "91748994d9b986ee501c7a4d11debaa3a7f13dc14c6a8cad7ff4703afe4b0007",
  "test_feedback_read": false,
  "files": {
    "validation_summary.csv": "967da7f181120facc12c9328fa39ff64ba22a749b7df4e975ab2091193b62606",
    "per_user_results.csv": "c96d8b31c420d7ef5

,model,users,mean_recall_at_10,mean_ndcg_at_10
0,EASE full catalog,100,0.143138,0.149123
1,IMDb hybrid full catalog,100,0.157483,0.150806
2,LLM direct full history,100,0.077828,0.093032


## Response validity, catalog coverage, and paired comparisons

Separate valid JSON generation from recommendation eligibility. A well-formed response can include
valid movies outside this benchmark. The audit exposes those cases rather than silently replacing
or dropping them. Paired bootstrap intervals compare the same users and do not adjust for prior
validation selection, prompt/model variance, or public-data pretraining contamination.

In [2]:
print("Attempted users:", report["attempted_users"], "/", report["planned_users"])
print("Valid-format responses:", report["valid_format_users"])
print("Estimated new token cost: $", round(report["estimated_token_cost_usd"], 6))
print("Shared conservative reservation, both experiments: $",
      round(report["cumulative_reserved_usd_both_experiments"], 6))
display(pd.DataFrame([{"status": key, "slots": value}
                      for key, value in report["slot_status_counts"].items()]))
if report["complete"]:
    audit = pd.read_csv(output_dir / "recommendation_audit.csv")
    paired = pd.read_csv(output_dir / "paired_bootstrap.csv")
    display(paired.round(6))
    example_users = sorted(audit["userId"].unique())[:2]
    display(audit.loc[audit["userId"].isin(example_users)])
else:
    print("Incomplete live run: the summary contains baselines only.")

Attempted users: 100 / 100
Valid-format responses: 100
Estimated new token cost: $ 0.021248
Shared conservative reservation, both experiments: $ 0.828731


,status,slots
0,eligible,910
1,unmatched_title_year,40
2,already_rated,34
3,outside_training_catalog,15
4,duplicate,1


,baseline,metric,mean_change,ci95_low,ci95_high,split_fingerprint,evaluation_split
0,EASE full catalog,recall_at_10,-0.065310,-0.107176,-0.025643,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
1,EASE full catalog,ndcg_at_10,-0.056090,-0.094082,-0.018595,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
2,IMDb hybrid full catalog,recall_at_10,-0.079655,-0.125723,-0.037754,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
3,IMDb hybrid full catalog,ndcg_at_10,-0.057774,-0.093671,-0.023773,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation


,userId,rank,generated_title,generated_year,matched_movieId,status,is_validation_like,split_fingerprint,evaluation_split
0,25,1,The Empire Strikes Back,1980,NaN,unmatched_title_year,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
1,25,2,Interstellar,2014,109487.0,eligible,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
2,25,3,The Dark Knight,2008,58559.0,eligible,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
3,25,4,The Fellowship of the Ring: Extended Edition,2002,NaN,unmatched_title_year,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
4,25,5,The Avengers,2012,89745.0,eligible,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
5,25,6,The Prestige,2006,48780.0,eligible,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
6,25,7,Spider-Man: Into the Spider-Verse,2018,NaN,unmatched_title_year,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
7,25,8,The Truman Show,1998,1682.0,eligible,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
8,25,9,The Green Mile,1999,3147.0,eligible,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
9,25,10,Jurassic Park,1993,480.0,eligible,False,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation


## Interpretation

This answers: **Given a user's ratings, how well can this LLM independently recover their recorded
future likes compared with models trained on MovieLens?** It is distinct from asking whether an
LLM can improve a shortlist produced by EASE.

Interpret ranking metrics together with catalog matching and seen/duplicate rates. A low score can
reflect preference errors, title/year matching errors, missing catalog coverage, or incomplete
relevance labels. Manual review of unmatched suggestions can assess the matcher; do not change
matching rules based on which changes increase validation hits. Freeze revisions for a new run.

All generated titles, request hashes, usage, and source provenance are cached under the ignored
`notebook12_llm_direct/` output directory. Rerunning the default notebook makes no API calls.